In [1]:
import pandas as pd
import random
from faker import Faker
fake = Faker()


In [2]:
#generation of random people and their pronouns,
pronoun_groups = [
    "she/her",
    "he/him",
    "they/them",
    "xe/xer",
    "she/they",
    "he/they"
    
]

number_of_people = 300

people = pd.DataFrame({
    "ID":range(1, number_of_people + 1),
    "Name": [fake.name() for _ in range(number_of_people)],
    "Pronouns": [random.choice(pronoun_groups) for _ in range(number_of_people)]
})



In [3]:
#extending generation process to have probabilitic parameters
#random connections now, extend the probabiltiies of pronouns

In [4]:
# Homophily-based network generation (affinity for similar pronouns)

connective_lst = []

MIN_CONNECTIONS = 3
MAX_CONNECTIONS = 8
SAME_PRONOUN_WEIGHT = 0.7   # probability of choosing same-pronoun person

for _, person in people.iterrows():
    person_id = person["ID"]
    person_pronoun = person["Pronouns"]

    number_of_connections = random.randint(MIN_CONNECTIONS, MAX_CONNECTIONS)

    # Split potential connections into similar vs different pronouns
    same_group = people[
        (people["Pronouns"] == person_pronoun) &
        (people["ID"] != person_id)
    ]["ID"].tolist()

    diff_group = people[
        (people["Pronouns"] != person_pronoun) &
        (people["ID"] != person_id)
    ]["ID"].tolist()

    chosen_ids = set()

    while len(chosen_ids) < number_of_connections:
        if random.random() < SAME_PRONOUN_WEIGHT and same_group:
            chosen_ids.add(random.choice(same_group))
        else:
            chosen_ids.add(random.choice(diff_group))

    for conid in chosen_ids:
        connective_lst.append({
            "FromID": person_id,
            "ToID": conid
        })

connections = pd.DataFrame(connective_lst)

In [5]:
#merging both people and pronouns into a singular excel sheet
merging = connections.merge(
    people, left_on="FromID", right_on="ID"
).merge(
    people, left_on="ToID", right_on="ID", suffixes=("_From", "_To")
)

final = merging[[
    "Name_From", "Pronouns_From",
    "Name_To", "Pronouns_To"
]]

final.head()

,Name_From,Pronouns_From,Name_To,Pronouns_To
0,Lisa Snyder,she/they,John Allen,he/him
1,Lisa Snyder,she/they,Nicole Patrick,she/they
2,Lisa Snyder,she/they,Nicole Stevens,she/they
3,Lisa Snyder,she/they,Logan Ponce,she/they
4,Monica Anderson,she/they,Erin Hatfield,she/they


In [6]:
# Copy user group for 
def map_user_group(pronoun_str):
    pronoun_str = pronoun_str.lower()
    if pronoun_str in ["he/him", "he"]:
        return "he"
    elif pronoun_str in ["she/her", "she"]:
        return "she"
    elif pronoun_str in ["they/them", "they"]:
        return "they"
    elif pronoun_str in ["xe/xer", "xe"]:
        return "xe"
    elif pronoun_str in ["she/they"]:
        return "she/they"
    elif pronoun_str in ["he/they"]:
        return "he/they"
    else:
        return pronoun_str  # keep unknown as is

people["User_Group"] = people["Pronouns"].apply(map_user_group)

# Each row will have a list of base pronouns
people["Base_Pronoun_List"] = people["User_Group"].apply(lambda x: x.split("/"))

# Explode so each base pronoun is a separate row
exploded = people.explode("Base_Pronoun_List")

# -------------------------------
# Step 5: Build Probability Matrix
# -------------------------------

pronoun_matrix = pd.crosstab(
    exploded["User_Group"],
    exploded["Base_Pronoun_List"],
    normalize="index"
)

# Ensure consistent column order
pronoun_matrix = pronoun_matrix.reindex(
    columns=["he", "she", "they", "xe"], fill_value=0
)

# Add user counts
group_counts = people["User_Group"].value_counts()
pronoun_matrix["User_Count"] = group_counts

# Move User_Count to first column
cols = ["User_Count"] + [col for col in pronoun_matrix.columns if col != "User_Count"]
pronoun_matrix = pronoun_matrix[cols]


In [7]:
pronoun_counts = people["Pronouns"].value_counts()
pronoun_prob = people["Pronouns"].value_counts(normalize=True)
pronoun_percent = pronoun_prob * 100

#connection pronoun probabilities
from_prob = final["Pronouns_From"].value_counts(normalize=True)
to_prob = final["Pronouns_To"].value_counts(normalize=True)

In [8]:
#excel export
final.to_csv("people_connections_to_pronouns_100.csv", index=False, encoding='utf-8-sig')
pronoun_matrix.to_csv("pronoun_affinity_constraints.csv")

